# NV single-projection reconstruction benchmark

This notebook benchmarks the reconstruction of the vector magnetic field from a single NV projection.

It does the following:

- builds several source maps
- computes the magnetic field for each map
- adds four field variants: no bias, in-plane bias, Bz bias, NV-axis bias
- generates NV projections
- creates noisy copies of the projections with Gaussian noise of 3 mT standard deviation
- runs the same grid search on all clean and noisy measurements
- compares the best hyperparameters for clean and noisy data
- plots ground truth, reconstruction and reconstruction error


In [ ]:
%matplotlib inline
from pathlib import Path
ROOT = Path.cwd()
assert (ROOT / "helpers.py").is_file(), "Launch from the repository root."
OUT = ROOT / "outputs" / "benchmark_all_cases"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from numpy.fft import fft2, ifft2, ifftshift
from scipy.signal import convolve2d
from scipy.ndimage import gaussian_filter
from itertools import product as iproduct
from matplotlib.patches import Patch
import pandas as pd

plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.titlesize'] = 10


In [ ]:

# Parameters

NX = NY = 110
Lx = Ly = 3e-6
dx = Lx / NX

z_meas = 100e-9
z_meas_idx = max(1, int(round(z_meas / dx)))
z_val = z_meas_idx * dx

Ms = 8e5
t_film = 10e-9
C_dip = 1e-7 * Ms * dx**2 * t_film

theta_nv = np.deg2rad(125.25)
phi_nv = np.deg2rad(90.0)
nv_hat = np.array([
    np.sin(theta_nv) * np.cos(phi_nv),
    np.sin(theta_nv) * np.sin(phi_nv),
    np.cos(theta_nv)
])
ux, uy, uz = nv_hat

B_bias_amp = 5e-3 #10e-3
B_bias_inplane = np.array([B_bias_amp, 0.0, 0.0])
B_bias_z = np.array([0.0, 0.0, B_bias_amp])
B_bias_nv = B_bias_amp * nv_hat

noise_sigma_mT = 3.0
noise_sigma_T = noise_sigma_mT * 1e-3
noise_seed = 1234

extent_um = [0, Lx * 1e6, 0, Ly * 1e6]

print(f"dx = {dx*1e9:.2f} nm")
print(f"z_meas = {z_val*1e9:.1f} nm (index {z_meas_idx})")
print(f"NV axis = [{ux:.4f}, {uy:.4f}, {uz:.4f}]")
print(f"Noise sigma on B_parallel = {noise_sigma_mT:.1f} mT")


In [ ]:

# Source maps

def generate_spin_lattice(nx, ny, n_domains=12, r_min=6, r_max=20, seed=4):
    rng = np.random.default_rng(seed)
    spins = rng.choice([-1.0, 1.0], size=(ny, nx), p=[0.48, 0.52]) * 0.1
    for _ in range(n_domains):
        cx, cy = rng.integers(0, nx), rng.integers(0, ny)
        r = rng.integers(r_min, r_max)
        val = rng.choice([-1.0, 1.0])
        yy, xx = np.ogrid[:ny, :nx]
        spins[(xx - cx)**2 + (yy - cy)**2 <= r**2] = val
    kernel = np.array([[0.5, 1.0, 0.5],
                       [1.0, 2.0, 1.0],
                       [0.5, 1.0, 0.5]]) / 9.0
    kernel /= kernel.sum()
    spins = np.sign(convolve2d(spins, kernel, mode='same', boundary='wrap') + 0.005)
    return spins

def make_smooth_gradient(nx, ny, axis='x', sigma_px=8):
    xv = np.linspace(-1, 1, nx)
    yv = np.linspace(-1, 1, ny)
    if axis == 'x':
        M = np.tile(xv, (ny, 1))
    elif axis == 'y':
        M = np.tile(yv[:, None], (1, nx))
    else:
        X, Y = np.meshgrid(xv, yv, indexing='xy')
        M = (X + Y) / np.sqrt(2)
    return gaussian_filter(M, sigma=sigma_px)

def make_gaussian_blob(nx, ny, sigma_frac=0.28):
    y, x = np.mgrid[0:ny, 0:nx]
    x0, y0 = (nx - 1) / 2, (ny - 1) / 2
    sigma = sigma_frac * min(nx, ny)
    M = np.exp(-((x - x0)**2 + (y - y0)**2) / (2 * sigma**2))
    return 2 * (M - M.min()) / (M.max() - M.min()) - 1

def make_cosine_mode(nx, ny, n_periods_x=1, n_periods_y=0):
    x = np.linspace(0, 2*np.pi*n_periods_x, nx, endpoint=False)
    y = np.linspace(0, 2*np.pi*n_periods_y, ny, endpoint=False)
    X, Y = np.meshgrid(x, y, indexing='xy')
    M = np.cos(X + Y)
    return M / np.max(np.abs(M))

def make_diffuse_bubble(nx, ny, radius_frac=0.23, wall_px=6):
    y, x = np.mgrid[0:ny, 0:nx]
    x0, y0 = (nx - 1) / 2, (ny - 1) / 2
    r = np.sqrt((x - x0)**2 + (y - y0)**2)
    R = radius_frac * min(nx, ny)
    return np.tanh((R - r) / wall_px)

def make_lowk_random(nx, ny, cutoff_frac=0.06, seed=0):
    rng = np.random.default_rng(seed)
    noise = rng.normal(size=(ny, nx))
    F = np.fft.fft2(noise)
    kx = np.fft.fftfreq(nx)
    ky = np.fft.fftfreq(ny)
    KX, KY = np.meshgrid(kx, ky, indexing='xy')
    K = np.sqrt(KX**2 + KY**2)
    H = np.exp(-(K / cutoff_frac)**2)
    M = np.real(np.fft.ifft2(F * H))
    M = (M - M.mean()) / (M.std() + 1e-12)
    return np.tanh(M / 1.5)

def make_stripes(nx, ny, n_periods=2, angle_deg=0, wall_px=5):
    y, x = np.mgrid[0:ny, 0:nx]
    x = x - (nx - 1) / 2
    y = y - (ny - 1) / 2
    th = np.deg2rad(angle_deg)
    U = x * np.cos(th) + y * np.sin(th)
    period = min(nx, ny) / n_periods
    return np.tanh(np.cos(2*np.pi * U / period) / (wall_px / 3))

source_maps = {
    'Domains': generate_spin_lattice(NX, NY),
    'Gaussian blob': make_gaussian_blob(NX, NY, sigma_frac=0.30),
    'Diffuse bubble': make_diffuse_bubble(NX, NY, radius_frac=0.24, wall_px=7),
    'Smooth gradient x': make_smooth_gradient(NX, NY, axis='x', sigma_px=10),
    'Low-k random': make_lowk_random(NX, NY, cutoff_frac=0.05, seed=1),
    'Wide stripes': make_stripes(NX, NY, n_periods=2, angle_deg=25, wall_px=7),
    'Cosine low-k': make_cosine_mode(NX, NY, n_periods_x=1, n_periods_y=0),
}


In [ ]:

# Show source maps

n_maps = len(source_maps)
ncols = 4
nrows = int(np.ceil(n_maps / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4.0 * nrows), constrained_layout=True)
axes = np.atleast_2d(axes)

for ax in axes.ravel():
    ax.axis('off')

for ax, (name, spins) in zip(axes.ravel(), source_maps.items()):
    ax.axis('on')
    im = ax.imshow(spins, origin='lower', extent=extent_um, cmap='viridis',
                   vmin=-1, vmax=1, interpolation='nearest')
    ax.set_title(name)
    ax.set_xlabel('x (μm)')
    ax.set_ylabel('y (μm)')
    fig.colorbar(im, ax=ax, label='source amplitude (dimensionless)', shrink=0.85)

plt.savefig(OUT / "source_maps.pdf")
plt.show()


In [ ]:

# Forward model

def dipole_kernels(nx, ny, dx, z, C):
    x = (np.arange(nx) - nx // 2) * dx
    y = (np.arange(ny) - ny // 2) * dx
    X, Y = np.meshgrid(x, y)
    R2 = X**2 + Y**2 + z**2
    R = np.sqrt(R2)
    with np.errstate(divide='ignore', invalid='ignore'):
        Kx = C * 3 * X * z / R2**2.5
        Ky = C * 3 * Y * z / R2**2.5
        Kz = C * (3 * z**2 / R2**2.5 - 1 / R2**1.5)
    Kx[R == 0] = 0.0
    Ky[R == 0] = 0.0
    Kz[R == 0] = 0.0
    return Kx, Ky, Kz

def field_at_z(spins, z, dx=dx, C=C_dip):
    ny, nx = spins.shape
    Kx, Ky, Kz = dipole_kernels(nx, ny, dx, z, C)
    S = fft2(spins)

    def conv(K):
        return np.real(ifft2(S * fft2(ifftshift(K))))

    return conv(Kx), conv(Ky), conv(Kz)

def nv_projection(Bx, By, Bz, nv=nv_hat):
    return Bx * nv[0] + By * nv[1] + Bz * nv[2]


In [ ]:

# Build all field cases and all measurements

bias_variants = {
    'No bias': np.array([0.0, 0.0, 0.0]),
    'In-plane bias': B_bias_inplane,
    'Bz bias': B_bias_z,
    'NV bias': B_bias_nv,
}

field_cases = {}
measurements = {}

rng_noise = np.random.default_rng(noise_seed)

for map_name, spins in source_maps.items():
    Bx0, By0, Bz0 = field_at_z(spins, z_val)
    Bmag0 = np.sqrt(Bx0**2 + By0**2 + Bz0**2)
    print(f"{map_name}: |B|_max = {Bmag0.max()*1e3:.3f} mT")

    for variant_name, bias_vec in bias_variants.items():
        case_name = f"{map_name} | {variant_name}"

        Bx = Bx0 + bias_vec[0]
        By = By0 + bias_vec[1]
        Bz = Bz0 + bias_vec[2]
        Bpar = nv_projection(Bx, By, Bz)

        field_cases[case_name] = {
            'map_name': map_name,
            'variant_name': variant_name,
            'spins': spins,
            'Bx': Bx,
            'By': By,
            'Bz': Bz,
            'Bpar_clean': Bpar,
        }

        measurements[(case_name, 'clean')] = Bpar
        measurements[(case_name, 'noisy')] = Bpar + rng_noise.normal(0.0, noise_sigma_T, size=Bpar.shape)

print()
print(f"Number of source maps: {len(source_maps)}")
print(f"Number of field cases: {len(field_cases)}")
print(f"Number of measurements: {len(measurements)}")


In [ ]:

# Quick table of field cases

rows = []
for case_name, case in field_cases.items():
    rows.append({
        'case': case_name,
        'mean Bx (mT)': case['Bx'].mean() * 1e3,
        'mean By (mT)': case['By'].mean() * 1e3,
        'mean Bz (mT)': case['Bz'].mean() * 1e3,
        'max |B| (mT)': np.sqrt(case['Bx']**2 + case['By']**2 + case['Bz']**2).max() * 1e3,
    })

df_cases = pd.DataFrame(rows)
print(df_cases.to_string(index=False))


## Reconstruction model

Forward model in k-space


$ \hat{B}_{\parallel}(\mathbf{k}) = D(\mathbf{k}) \hat{B}_z(\mathbf{k}), \qquad
D(\mathbf{k}) = u_z - i \frac{u_x k_x + u_y k_y}{k} $

Inversion

$ \hat{B}_z =\frac{D^*(\mathbf{k})}{|D|^2 + \lambda(k)} \hat{B}_{\parallel}, \qquad
\hat{B}_{x,y} = -i \frac{k_{x,y}}{k}\hat{B}_z $

Regularisation


$\lambda(k) = \lambda_0 \left(1 + \left(\frac{k}{k_0}\right)^p\right),
\qquad
k_0 =\alpha / z_{\mathrm{meas}}$

At k = 0:
- Bz DC is assigned B_parallel / uz by convention; this is not an absolute measurement
- Bx DC and By DC cannot be recovered from a single projection and are set to zero


In [ ]:

# Reconstruction operators

kx_arr = 2 * np.pi * np.fft.fftfreq(NX, d=dx)
ky_arr = 2 * np.pi * np.fft.fftfreq(NY, d=dx)
KX, KY = np.meshgrid(kx_arr, ky_arr, indexing='xy')
K = np.sqrt(KX**2 + KY**2)
K_safe = np.where(K == 0, 1.0, K)

D_op = uz - 1j * (ux * KX + uy * KY) / K_safe
D2 = np.abs(D_op)**2

def reconstruct(Bpar_hat, p, alpha_k0, lam0_fac):
    k0 = alpha_k0 / max(z_val, 1e-12)
    lam_k = lam0_fac * (1.0 + (K / k0) ** p)

    Bz_hat = np.conj(D_op) * Bpar_hat / (D2 + lam_k)

    if abs(uz) > 1e-10:
        Bz_hat[0, 0] = Bpar_hat[0, 0] / uz

    Bx_hat = -1j * (KX / K_safe) * Bz_hat
    By_hat = -1j * (KY / K_safe) * Bz_hat
    Bx_hat[0, 0] = 0.0
    By_hat[0, 0] = 0.0

    return (
        np.real(ifft2(Bx_hat)),
        np.real(ifft2(By_hat)),
        np.real(ifft2(Bz_hat))
    )

def rmse_mT(A, B):
    return float(np.sqrt(np.mean((np.asarray(A) - np.asarray(B))**2))) * 1e3


In [ ]:

# Grid search setup

p_list = [1, 2, 4, 7, 10, 15, 20]
alpha_k0_list = [0.5, 1.0, 2.0, 3.0, 6.0, 10.0]
lam0_factors = np.logspace(-20, 0, 15)

print(f"p values: {p_list}")
print(f"alpha_k0 values: {alpha_k0_list}")
print(f"lambda_0 values: {lam0_factors}")
print(f"Total configs per measurement: {len(p_list) * len(alpha_k0_list) * len(lam0_factors)}")


In [ ]:

# Run grid search on all clean and noisy measurements

all_results = {}
all_best_rec = {}
best_rows = []

for case_name, case in field_cases.items():
    gt_Bx = case['Bx']
    gt_By = case['By']
    gt_Bz = case['Bz']

    for meas_label in ['clean', 'noisy']:
        Bpar = measurements[(case_name, meas_label)]
        Bpar_hat = fft2(Bpar)

        rows = []
        for p, alpha_k0, lam0_fac in iproduct(p_list, alpha_k0_list, lam0_factors):
            Bx_r, By_r, Bz_r = reconstruct(Bpar_hat, p, alpha_k0, lam0_fac)

            ex = rmse_mT(Bx_r, gt_Bx)
            ey = rmse_mT(By_r, gt_By)
            ez = rmse_mT(Bz_r, gt_Bz)
            score = (ex + ey + ez) / 3.0

            rows.append({
                'case': case_name,
                'map_name': case['map_name'],
                'variant_name': case['variant_name'],
                'measurement': meas_label,
                'p': p,
                'alpha_k0': alpha_k0,
                'lam0_fac': lam0_fac,
                'RMSE_Bx': ex,
                'RMSE_By': ey,
                'RMSE_Bz': ez,
                'score': score,
            })

        df = pd.DataFrame(rows).sort_values('score').reset_index(drop=True)
        all_results[(case_name, meas_label)] = df

        best = df.iloc[0]
        best_rec = reconstruct(Bpar_hat, int(best.p), float(best.alpha_k0), float(best.lam0_fac))
        all_best_rec[(case_name, meas_label)] = best_rec

        best_rows.append({
            'case': case_name,
            'map_name': case['map_name'],
            'variant_name': case['variant_name'],
            'measurement': meas_label,
            'p': int(best.p),
            'alpha_k0': float(best.alpha_k0),
            'lam0_fac': float(best.lam0_fac),
            'RMSE_Bx': float(best.RMSE_Bx),
            'RMSE_By': float(best.RMSE_By),
            'RMSE_Bz': float(best.RMSE_Bz),
            'score': float(best.score),
        })

        print(f"{case_name} | {meas_label}")
        print(f"  best p = {int(best.p)}")
        print(f"  best alpha_k0 = {best.alpha_k0}")
        print(f"  best lambda_0 = {best.lam0_fac:.2e}")
        print(f"  RMSE Bx = {best.RMSE_Bx:.3f} mT")
        print(f"  RMSE By = {best.RMSE_By:.3f} mT")
        print(f"  RMSE Bz = {best.RMSE_Bz:.3f} mT")
        print(f"  mean RMSE = {best.score:.3f} mT")
        print()


In [ ]:

# Summary tables

df_best = pd.DataFrame(best_rows)
print("Best configuration for each case")
print(df_best.to_string(index=False))

def geom_mean(x):
    x = np.asarray(x, dtype=float)
    x = x[x > 0]
    return float(np.exp(np.mean(np.log(x)))) if len(x) else np.nan

avg_params = (
    df_best.groupby('measurement')
    .agg(
        p_mean=('p', 'mean'),
        p_median=('p', 'median'),
        alpha_k0_mean=('alpha_k0', 'mean'),
        alpha_k0_median=('alpha_k0', 'median'),
        lam0_fac_geom_mean=('lam0_fac', geom_mean),
        lam0_fac_median=('lam0_fac', 'median'),
        RMSE_Bx_mean=('RMSE_Bx', 'mean'),
        RMSE_By_mean=('RMSE_By', 'mean'),
        RMSE_Bz_mean=('RMSE_Bz', 'mean'),
        score_mean=('score', 'mean'),
    )
    .reset_index()
)

print()
print("Average hyperparameters by measurement type")
print(avg_params.to_string(index=False))

mode_rows = []
for meas_label, subdf in df_best.groupby('measurement'):
    mode_rows.append({
        'measurement': meas_label,
        'p_mode': subdf['p'].mode().iloc[0],
        'alpha_k0_mode': subdf['alpha_k0'].mode().iloc[0],
        'lam0_fac_mode': subdf['lam0_fac'].mode().iloc[0],
    })

df_modes = pd.DataFrame(mode_rows)
print()
print("Most frequent best parameters by measurement type")
print(df_modes.to_string(index=False))


In [ ]:

# Plot helper

QSTEP = 6
xs_q = np.linspace(0, Lx * 1e6, NX)[::QSTEP]
ys_q = np.linspace(0, Ly * 1e6, NY)[::QSTEP]
Xs_q, Ys_q = np.meshgrid(xs_q, ys_q)

def plot_field_quiver(ax, Bx, By, Bz, title, ref_arrow, vmax_mT, cmap='bwr'):
    im = ax.imshow(Bz * 1e3, origin='lower', extent=extent_um,
                   cmap=cmap, vmin=-vmax_mT, vmax=vmax_mT, interpolation='nearest')
    U = Bx[::QSTEP, ::QSTEP]
    V = By[::QSTEP, ::QSTEP]
    ax.quiver(Xs_q, Ys_q, U / ref_arrow, V / ref_arrow,
              color='k', alpha=0.95, scale=18, width=0.004, headwidth=3.5)
    ax.set_title(title)
    ax.set_xlabel('x (μm)')
    ax.set_ylabel('y (μm)')
    return im


In [ ]:

# Plot ground truth vs best reconstruction for all clean and noisy measurements

for case_name, case in field_cases.items():
    gt_Bx = case['Bx']
    gt_By = case['By']
    gt_Bz = case['Bz']

    for meas_label in ['clean', 'noisy']:
        Bx_r, By_r, Bz_r = all_best_rec[(case_name, meas_label)]

        dBx = Bx_r - gt_Bx
        dBy = By_r - gt_By
        dBz = Bz_r - gt_Bz

        mag_gt = np.sqrt(gt_Bx[::QSTEP, ::QSTEP]**2 + gt_By[::QSTEP, ::QSTEP]**2)
        ref_main = np.percentile(mag_gt[mag_gt > 0], 90) if (mag_gt > 0).any() else 1.0
        ref_main = max(ref_main, 1e-12)

        mag_diff = np.sqrt(dBx[::QSTEP, ::QSTEP]**2 + dBy[::QSTEP, ::QSTEP]**2)
        ref_diff = np.percentile(mag_diff[mag_diff > 0], 90) if (mag_diff > 0).any() else ref_main
        ref_diff = max(ref_diff, 1e-12)

        arrow_zoom = ref_main / ref_diff

        vmax_main = max(np.abs(gt_Bz).max(), np.abs(Bz_r).max()) * 1e3
        vmax_main = max(vmax_main, 1e-9)
        vmax_diff = max(np.abs(dBz).max() * 1e3, 1e-9)

        rmse_bx = rmse_mT(Bx_r, gt_Bx)
        rmse_by = rmse_mT(By_r, gt_By)
        rmse_bz = rmse_mT(Bz_r, gt_Bz)

        fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8), constrained_layout=True)
        fig.suptitle(
            f"{case_name} | measurement = {meas_label}\n"
            f"RMSE Bx = {rmse_bx:.3f} mT, RMSE By = {rmse_by:.3f} mT, RMSE Bz = {rmse_bz:.3f} mT",
            fontsize=12
        )

        im0 = plot_field_quiver(
            axes[0], gt_Bx, gt_By, gt_Bz,
            title='Ground truth',
            ref_arrow=ref_main,
            vmax_mT=vmax_main
        )

        im1 = plot_field_quiver(
            axes[1], Bx_r, By_r, Bz_r,
            title='Reconstructed',
            ref_arrow=ref_main,
            vmax_mT=vmax_main
        )

        im2 = plot_field_quiver(
            axes[2], dBx, dBy, dBz,
            title='Difference (recon - GT)',
            ref_arrow=ref_diff,
            vmax_mT=vmax_diff
        )

        axes[2].legend(
            handles=[Patch(facecolor='none', edgecolor='none',
                           label=f'Quiver x{arrow_zoom:.1f}')],
            loc='upper left',
            framealpha=0.9,
            handlelength=0,
            handletextpad=0
        )

        cbar_main = fig.colorbar(im0, ax=axes[:2], shrink=0.88)
        cbar_main.set_label('Bz (mT)')

        cbar_diff = fig.colorbar(im2, ax=axes[2], shrink=0.88)
        cbar_diff.set_label('Delta Bz (mT)')

        plt.show()


In [ ]:

# Final summary

summary_rows = []
for (case_name, meas_label), df in all_results.items():
    best = df.iloc[0]
    summary_rows.append({
        'case': case_name,
        'measurement': meas_label,
        'p': int(best.p),
        'alpha_k0': best.alpha_k0,
        'lam0_fac': best.lam0_fac,
        'RMSE_Bx (mT)': best.RMSE_Bx,
        'RMSE_By (mT)': best.RMSE_By,
        'RMSE_Bz (mT)': best.RMSE_Bz,
        'mean RMSE (mT)': best.score,
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))

print()
print("k = 0 limitation")
print(f"ux = {ux:.4f}, uy = {uy:.4f}, uz = {uz:.4f}")
print(f"Bz DC recoverable because uz != 0")
print(f"Bx DC not recoverable from a single NV projection")
print(f"By DC not recoverable from a single NV projection")


In [ ]:
# Average best hyperparameters for clean and noisy measurements

def geom_mean(x):
    x = np.asarray(x, dtype=float)
    x = x[x > 0]
    return float(np.exp(np.mean(np.log(x)))) if len(x) else np.nan

best_rows = []
for (case_name, meas_label), df in all_results.items():
    best = df.iloc[0]
    best_rows.append({
        'case': case_name,
        'measurement': meas_label,
        'p': int(best.p),
        'alpha_k0': float(best.alpha_k0),
        'lam0_fac': float(best.lam0_fac),
        'RMSE_Bx_mT': float(best.RMSE_Bx),
        'RMSE_By_mT': float(best.RMSE_By),
        'RMSE_Bz_mT': float(best.RMSE_Bz),
        'score_mT': float(best.score),
    })

df_best = pd.DataFrame(best_rows)

df_avg = (
    df_best.groupby('measurement')
    .agg(
        p_mean=('p', 'mean'),
        p_median=('p', 'median'),
        alpha_k0_mean=('alpha_k0', 'mean'),
        alpha_k0_median=('alpha_k0', 'median'),
        lam0_fac_geom_mean=('lam0_fac', geom_mean),
        lam0_fac_median=('lam0_fac', 'median'),
        RMSE_Bx_mean_mT=('RMSE_Bx_mT', 'mean'),
        RMSE_By_mean_mT=('RMSE_By_mT', 'mean'),
        RMSE_Bz_mean_mT=('RMSE_Bz_mT', 'mean'),
        score_mean_mT=('score_mT', 'mean'),
    )
    .reset_index()
)

print("Average best hyperparameters by measurement type")
print(df_avg.to_string(index=False))

In [ ]:
# Average best hyperparameters
# Global: clean vs noisy
# Clean only: No bias vs NV bias

def geom_mean(x):
    x = np.asarray(x, dtype=float)
    x = x[x > 0]
    return float(np.exp(np.mean(np.log(x)))) if len(x) else np.nan

best_rows = []
for (case_name, meas_label), df in all_results.items():
    best = df.iloc[0]

    # case_name format: "Map name | Variant name"
    if " | " in case_name:
        map_name, variant_name = case_name.split(" | ", 1)
    else:
        map_name, variant_name = case_name, "unknown"

    best_rows.append({
        'case': case_name,
        'map_name': map_name,
        'variant_name': variant_name,
        'measurement': meas_label,
        'p': int(best.p),
        'alpha_k0': float(best.alpha_k0),
        'lam0_fac': float(best.lam0_fac),
        'RMSE_Bx_mT': float(best.RMSE_Bx),
        'RMSE_By_mT': float(best.RMSE_By),
        'RMSE_Bz_mT': float(best.RMSE_Bz),
        'score_mT': float(best.score),
    })

df_best = pd.DataFrame(best_rows)

def summarize_best(df_in, title):
    df_out = (
        df_in.groupby(df_in.columns[0])
        .agg(
            p_mean=('p', 'mean'),
            p_median=('p', 'median'),
            alpha_k0_mean=('alpha_k0', 'mean'),
            alpha_k0_median=('alpha_k0', 'median'),
            lam0_fac_geom_mean=('lam0_fac', geom_mean),
            lam0_fac_median=('lam0_fac', 'median'),
            RMSE_Bx_mean_mT=('RMSE_Bx_mT', 'mean'),
            RMSE_By_mean_mT=('RMSE_By_mT', 'mean'),
            RMSE_Bz_mean_mT=('RMSE_Bz_mT', 'mean'),
            score_mean_mT=('score_mT', 'mean'),
        )
        .reset_index()
    )
    print(title)
    print(df_out.to_string(index=False))
    print()
    return df_out

# 1) Global averages: clean vs noisy
df_global = summarize_best(
    df_best[['measurement', 'p', 'alpha_k0', 'lam0_fac',
             'RMSE_Bx_mT', 'RMSE_By_mT', 'RMSE_Bz_mT', 'score_mT']],
    "Average best hyperparameters by measurement type"
)

# 2) Clean only: compare No bias vs NV bias
df_clean_bias = df_best[
    (df_best['measurement'] == 'clean') &
    (df_best['variant_name'].isin(['No bias', 'NV bias']))
].copy()

df_clean_bias_avg = summarize_best(
    df_clean_bias[['variant_name', 'p', 'alpha_k0', 'lam0_fac',
                   'RMSE_Bx_mT', 'RMSE_By_mT', 'RMSE_Bz_mT', 'score_mT']],
    "Clean data only: average best hyperparameters for No bias vs NV bias"
)

### Fixed protocol and interpretation

The oracle grid search selects parameters using the ground truth of each evaluated map.
Its Gaussian noise (3 mT, one realization per case) is an illustrative stress test,
not an experimental noise model or an uncertainty estimate. Averaged best parameters
do not define a separately validated configuration. The fixed reference protocol is
lambda0=1e-8, p=10, alpha=1, with no median scaling. The DC convention in these
diagnostic benchmarks assigns the mean projection to Bz; compare AC fields when
evaluating spatial reconstruction. The sign-recovery and experimental notebooks use
the same spectral filter and zero reconstructed means.


In [ ]:
# Evaluate fixed parameters independently of the oracle parameter selection.
from sim_odmr_field_helpers import make_error_dataframe
fixed_rows = []
if 'field_cases' in globals():
    cases_fixed = {name: (case['Bx'], case['By'], case['Bz'])
                   for name, case in field_cases.items()}
else:
    cases_fixed = gt_fields
for name, truth in cases_fixed.items():
    projection = nv_projection(*truth)
    reconstruction = reconstruct(fft2(projection), 10, 1.0, 1e-8)
    metrics = make_error_dataframe(projection*1e3, projection*1e3, reconstruction, truth)
    metrics.insert(0, 'case', name)
    fixed_rows.append(metrics)
fixed_metrics = pd.concat(fixed_rows, ignore_index=True)
display(fixed_metrics)


In [ ]:
fixed_metrics.to_csv(OUT / "fixed_protocol_metrics.csv", index=False)
df_best.to_csv(OUT / "oracle_best_parameters.csv", index=False)
df_global.to_csv(OUT / "oracle_clean_noisy_summary.csv", index=False)
df_clean_bias_avg.to_csv(OUT / "oracle_clean_bias_summary.csv", index=False)
